In [1]:
# ! pip install -q --upgrade langchain langchain-openai langchain-core langchain_community docx2txt pypdf  langchain_chroma sentence_transformers

### **Part 1:**

- **Understanding RAG (Retrieval-Augmented Generation)**
    - What is RAG?
    - Importance of RAG in AI applications
- **Introduction to LangChain and LCEL**
    - Overview of LangChain
    - LangChain Expression Language (LCEL)
- **Exploring LangChain Components**
    - LLM (Large Language Models)
    - Prompts
    - Retrievers
    - Composing components into a chain
- **Document Processing and Vector Databases**
    - Document splitting techniques
    - Embedding documents
    - Storing and retrieving documents in a vector database

- **Creating Your First RAG Chain**
    - Step-by-step guide to building a basic RAG chain
    - Answering questions from documents using RAG
- **Building a Conversational RAG**
    - Handling follow-up questions
    - Concept of contextualizing and refining queries
- **Using pre-built LangChain RAG chains**
    - history_aware_retriever
    - create_retrieval_chain
- **Building Multi User Chatbot**
    - Managing conversation history using a database table


### What is Retrieval Augmented Generation (RAG)?
RAG is a technique that enhances language models by combining them with a retrieval system. It allows the model to access and utilize external knowledge when generating responses.

The process typically involves:
#### Indexing a large corpus of documents

In [2]:
from pprint import pprint as pp

In [3]:
# check and setup environment variables and API keys for Langchain and OpenAI

import os
import dotenv

dotenv.load_dotenv()

# LANGCHAIN_PROJECT to set the project name for Langchain
print("LANGCHAIN_PROJECT:", os.environ.get("LANGCHAIN_PROJECT", "langchain-RAG"))
# LANGCHAIN_TRACING to keep track of the execution
print("LANGCHAIN_TRACING:", os.environ.get("LANGCHAIN_TRACING_V2", "Not set"))
# LANGSMITH_ENDPOINT to set the endpoint for Langsmith
print("Langsmith Endpoint:", os.environ.get("LANGSMITH_ENDPOINT", "Not set"))
# OPENAI_API_KEY to set the API key for OpenAI
print("OpenAI API Key:", os.environ.get("OPENAI_API_KEY", "No API Key found")[:10])
# LANGCHAIN_API_KEY to set the API key for Langchain
print("Langchain API Key:", os.environ.get("LANGCHAIN_API_KEY", "No API Key found")[:10])

LANGCHAIN_PROJECT: langchain-RAG
LANGCHAIN_TRACING: true
Langsmith Endpoint: https://api.smith.langchain.com
OpenAI API Key: sk-proj-Wj
Langchain API Key: lsv2_pt_70


### Call LLM

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm_response = llm.invoke("Tell me a joke")

llm_response

AIMessage(content='Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 11, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': None, 'id': 'chatcmpl-BvJao4axdxW5nLIckYO5CqvYdvqXQ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d33bda6d-c4dc-40c9-9d92-a2aa23b95843-0', usage_metadata={'input_tokens': 11, 'output_tokens': 18, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Parsing Output

In [5]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

'Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!'

### Simple Chain

In [6]:
chain = llm | output_parser
chain.invoke("Tell me a joke")

"Why don't skeletons fight each other? \n\nThey don't have the guts!"

### Structured Output

In [7]:
from typing import List
from pydantic import BaseModel, Field


class MobileReview(BaseModel):
    phone_model: str = Field(description="Name and model of the phone")
    rating: float = Field(description="Overall rating out of 5")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    summary: str = Field(description="Brief summary of the review")


In [8]:
review_text = """
Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
stronger. Battery life's solid, lasts me all day no problem.

Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
Also, still getting used to the new button layout, keep hitting Bixby by mistake.

Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
being perfect. If you're due for an upgrade, definitely worth checking out!
"""

# Create a structured LLM with the MobileReview output parser
structured_llm = llm.with_structured_output(MobileReview)
output = structured_llm.invoke(review_text)
print(output)

phone_model='Samsung Galaxy S21' rating=4.0 pros=['Gorgeous screen with vibrant colors', 'Insane camera performance, especially at night', 'Solid battery life lasts all day'] cons=['Quite pricey', 'No charger included in the box', 'New button layout takes getting used to'] summary='Overall, a solid phone with impressive features, but some drawbacks like price and missing charger prevent it from being perfect.'


In [9]:
pp(dict(output))

{'cons': ['Quite pricey',
          'No charger included in the box',
          'New button layout takes getting used to'],
 'phone_model': 'Samsung Galaxy S21',
 'pros': ['Gorgeous screen with vibrant colors',
          'Insane camera performance, especially at night',
          'Solid battery life lasts all day'],
 'rating': 4.0,
 'summary': 'Overall, a solid phone with impressive features, but some '
            'drawbacks like price and missing charger prevent it from being '
            'perfect.'}


In [10]:
output.pros

['Gorgeous screen with vibrant colors',
 'Insane camera performance, especially at night',
 'Solid battery life lasts all day']

### Prompt Template

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
prompt.invoke({"topic": "programming"})

ChatPromptValue(messages=[HumanMessage(content='Tell me a short joke about programming', additional_kwargs={}, response_metadata={})])

In [12]:
chain = prompt | llm  # | output_parser
response = chain.invoke({"topic": "programer"})
pp(response)

AIMessage(content='Why do programmers prefer dark mode?\n\nBecause light attracts bugs!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 15, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': None, 'id': 'chatcmpl-BvJasorxxAOZrwoK476GvdBrA25xe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--1afd9d07-3e94-4479-8c66-4b41a99c3f26-0', usage_metadata={'input_tokens': 15, 'output_tokens': 12, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


In [13]:
chain = prompt | llm | output_parser
chain.invoke({"topic": "programer"})

'Why do programmers prefer dark mode?\n\nBecause light attracts bugs!'

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define the prompt
prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Define the output parser
output_parser = StrOutputParser()

# Compose the chain
chain = prompt | llm | output_parser

# Use the chain
result = chain.invoke({"topic": "programming"})
print(result)

Why do programmers prefer dark mode? 

Because light attracts bugs!


### LLM Messages
- System and User messages to provide persona to LLM

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content="You are a helpful assistant that tells jokes.")
human_message = HumanMessage(content="Tell me about programming")
llm.invoke([system_message, human_message])

AIMessage(content='Programming is a fascinating and creative process where you write code to instruct computers to perform specific tasks. It involves using various programming languages, like Python, Java, or C++, to solve problems, automate processes, or create applications. \n\nAnd speaking of programming, did you hear about the programmer who had a problem with his code? \n\nHe couldn’t find the "bugs" because every time he opened the window, he let them out! 🐛💻 \n\nIf you have specific topics in programming you want to know about, feel free to ask!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 24, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': None, 'id': 'chatcmpl-BvJatFFW

In [16]:
template = ChatPromptTemplate(
    [
        ("system", "You are a helpful assistant that tells jokes."),
        ("human", "Tell me about {topic}"),
    ]
)

prompt_value = template.invoke({"topic": "programming"})
prompt_value

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that tells jokes.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Tell me about programming', additional_kwargs={}, response_metadata={})])

In [17]:
llm.invoke(prompt_value)

AIMessage(content='Programming is the process of creating a set of instructions that a computer can follow to perform specific tasks. Programmers use various programming languages, like Python, Java, or C++, to write code that tells computers what to do. \n\nHere’s a programming-themed joke for you:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 24, 'total_tokens': 91, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': None, 'id': 'chatcmpl-BvJavHmEcqzYhXOiwF1LOJ9tNaJTw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d147563d-ffe4-4861-b790-bded1b33aa2c-0', usage_metadata={'input_tokens': 24, 'output_

### Download documents for RAG

In [18]:
# !pip install docx2txt pypdf unstructured

In [19]:
# Download docs

import requests
from pathlib import Path


def download_file_from_github(raw_url, save_path):
    response = requests.get(raw_url)
    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)
        print(f"File downloaded successfully and saved to {save_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")


# Example:
files_list = [
    # "https://raw.githubusercontent.com/karanpratapsingh/langchain-RAG/main/docs/Programming with Python.docx",
    "https://github.com/PradipNichite/Youtube-Tutorials/raw/refs/heads/main/Langchain RAG Course 2024/docs/GreenGrow's EcoHarvest System_ A Revolution in Farming.pdf",
    "https://github.com/PradipNichite/Youtube-Tutorials/raw/refs/heads/main/Langchain RAG Course 2024/docs/GreenGrow Innovations_ Company History.docx",
    "https://github.com/PradipNichite/Youtube-Tutorials/raw/refs/heads/main/Langchain RAG Course 2024/docs/Company_ TechWave Innovations.docx",
    "https://github.com/PradipNichite/Youtube-Tutorials/raw/refs/heads/main/Langchain RAG Course 2024/docs/Company_ QuantumNext Systems.docx",
    "https://github.com/PradipNichite/Youtube-Tutorials/raw/refs/heads/main/Langchain RAG Course 2024/docs/Company_ GreenFields BioTech.docx",
]

docs_path = Path("docs")
docs_path.mkdir(exist_ok=True)

for file_url in files_list:
    file_name = Path(file_url).name
    save_to = docs_path / file_name
    download_file_from_github(file_url, save_to)


File downloaded successfully and saved to docs\GreenGrow's EcoHarvest System_ A Revolution in Farming.pdf
File downloaded successfully and saved to docs\GreenGrow Innovations_ Company History.docx
File downloaded successfully and saved to docs\Company_ TechWave Innovations.docx
File downloaded successfully and saved to docs\Company_ QuantumNext Systems.docx
File downloaded successfully and saved to docs\Company_ GreenFields BioTech.docx


### Split Documents

In [20]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from typing import List
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)


In [21]:
docx_loader = Docx2txtLoader("./docs/GreenGrow Innovations_ Company History.docx")
documents = docx_loader.load()

print(len(documents))

splits = text_splitter.split_documents(documents)

print(f"Split the documents into {len(splits)} chunks.")

1
Split the documents into 2 chunks.


In [22]:
documents[0]

Document(metadata={'source': './docs/GreenGrow Innovations_ Company History.docx'}, page_content="GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analyz

In [23]:
splits[1]

Document(metadata={'source': './docs/GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultural techn

In [24]:
splits[0].metadata

{'source': './docs/GreenGrow Innovations_ Company History.docx'}

In [25]:
splits[0].page_content

'GreenGrow Innovations was founded in 2010 by Sarah Chen and Michael Rodriguez, two agricultural engineers with a passion for sustainable farming. The company started in a small garage in Portland, Oregon, with a simple mission: to make farming more environmentally friendly and efficient.\n\n\n\nIn its early days, GreenGrow focused on developing smart irrigation systems that could significantly reduce water usage in agriculture. Their first product, the WaterWise Sensor, was launched in 2012 and quickly gained popularity among local farmers. This success allowed the company to expand its research and development efforts.\n\n\n\nBy 2015, GreenGrow had outgrown its garage origins and moved into a proper office and research facility in the outskirts of Portland. This move coincided with the development of their second major product, the SoilHealth Monitor, which used advanced sensors to analyze soil composition and provide real-time recommendations for optimal crop growth.'

In [26]:
# import nltk
# nltk.download('punkt')

### Chunk Docs and create embeddings

In [27]:
# 1. Function to load documents from a folder


def load_documents(folder_path: str) -> List[Document]:
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        elif filename.endswith(".docx"):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents


# Load documents from a folder
folder_path = Path("./docs")
documents = load_documents(folder_path)

print(f"Loaded {len(documents)} documents from the folder.")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")

Loaded 5 documents from the folder.
Split the documents into 8 chunks.


#### Create embeddings using OpenAI Embeddings

In [28]:
embeddings = OpenAIEmbeddings()

# 4. Embedding Documents

document_embeddings = embeddings.embed_documents([split.page_content for split in splits])

print(f"Created embeddings for {len(document_embeddings)} document chunks.")

Created embeddings for 8 document chunks.


In [29]:
document_embeddings

[[-0.00023503396369051188,
  -0.024361496791243553,
  -0.009824666194617748,
  -0.011334133334457874,
  -0.016407256945967674,
  0.01967558264732361,
  -0.030556876212358475,
  -0.002741652773693204,
  -0.012272628955543041,
  -0.01542282197624445,
  0.005115782842040062,
  -0.014254625886678696,
  -0.014241499826312065,
  -0.009667156264185905,
  0.012902667745947838,
  0.008846793323755264,
  0.016801031306385994,
  -0.02570032887160778,
  -0.018966790288686752,
  0.009686845354735851,
  -0.031974464654922485,
  0.03417959809303284,
  0.011806662194430828,
  0.0020328592509031296,
  0.01596098020672798,
  0.015974106267094612,
  0.005466898437589407,
  -0.01753607764840126,
  -0.013887102715671062,
  -0.005512838717550039,
  -0.023783961310982704,
  0.005414395127445459,
  -0.03323454037308693,
  0.029034283012151718,
  -0.023206425830721855,
  -0.018376128748059273,
  0.0040624369867146015,
  -0.010290632024407387,
  0.013532706536352634,
  -0.005827858112752438,
  0.030189353972673

#### Create embeddings using Sentence transformers

In [30]:
# !pip install sentence_transformers
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_community.embeddings.sentence_transformer import (
#     SentenceTransformerEmbeddings,
# )

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # show_progress=True,
)


document_embeddings = embedding_function.embed_documents(
    [split.page_content for split in splits]
)
document_embeddings[0]

[0.02933572605252266,
 -0.02779753878712654,
 -0.01846339739859104,
 -0.05892423540353775,
 0.08951884508132935,
 -0.026082245633006096,
 -0.10806197673082352,
 0.037593018263578415,
 -0.005008626263588667,
 -0.041687238961458206,
 -0.016387756913900375,
 -0.026129279285669327,
 -0.03135998919606209,
 0.010636316612362862,
 -0.07282692939043045,
 -0.003014781279489398,
 -0.018765762448310852,
 -0.01873072236776352,
 0.06654553860425949,
 -0.07033523917198181,
 0.0032460829243063927,
 -0.01593751087784767,
 0.06552289426326752,
 0.014457402750849724,
 -0.011876796372234821,
 0.10337898135185242,
 -0.00531398830935359,
 0.0017421678639948368,
 0.0036550431977957487,
 -0.036102164536714554,
 -0.012576746754348278,
 0.04590792953968048,
 0.030862489715218544,
 -0.05400511249899864,
 0.026347419247031212,
 0.08197520673274994,
 -0.03140779212117195,
 -0.048109326511621475,
 0.049225907772779465,
 -0.002444606274366379,
 0.005876438692212105,
 -0.014024590142071247,
 -0.0039660208858549595,


### Create and persist Chroma vector store

In [31]:
from langchain_chroma import Chroma

embedding_function = OpenAIEmbeddings()
collection_name = "my_collection"
vectorstore = Chroma.from_documents(
    collection_name=collection_name,
    documents=splits,
    embedding=embedding_function,
    persist_directory="./chroma_db",
)
# db.persist()

print("Vector store created and persisted to './chroma_db'")

Vector store created and persisted to './chroma_db'


In [32]:
# 5. Perform similarity search

query = "When was GreenGrow Innovations founded?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: '{query}'\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"Source: {result.metadata.get('source', 'Unknown')}")
    print(f"Content: {result.page_content}")
    print()


Top 2 most relevant chunks for the query: 'When was GreenGrow Innovations founded?'

Result 1:
Source: docs\GreenGrow Innovations_ Company History.docx
Content: The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.



Today, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.



Despite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research

In [33]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever.invoke("When was GreenGrow Innovations founded?")

[Document(id='501eaaeb-ea42-4d73-aa60-147a8ab65f68', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions

In [34]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(template)

In [35]:
from langchain.schema.runnable import RunnablePassthrough

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt
rag_chain.invoke("When was GreenGrow Innovations founded?")

ChatPromptValue(messages=[HumanMessage(content='Answer the question based only on the following context:\n[Document(id=\'501eaaeb-ea42-4d73-aa60-147a8ab65f68\', metadata={\'source\': \'docs\\\\GreenGrow Innovations_ Company History.docx\'}, page_content="The company\'s breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\\n\\n\\n\\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\\n\\n\\n\\nDespite its growth, GreenGrow remains committed to its original mis

In [36]:
def docs2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [37]:
rag_chain = {"context": retriever | docs2str, "question": RunnablePassthrough()} | prompt
rag_chain.invoke("When was GreenGrow Innovations founded?")

ChatPromptValue(messages=[HumanMessage(content="Answer the question based only on the following context:\nThe company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions to advance the field of agricultur

In [38]:
rag_chain = (
    {"context": retriever | docs2str, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
question = "When was GreenGrow Innovations founded?"
response = rag_chain.invoke(question)
print(response)

GreenGrow Innovations was founded in 2010.


### Conversational RAG



#### Handling Follow Up Questions

In [39]:
# Example conversation
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response),
    ]
)

In [40]:
chat_history

[HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={})]

In [41]:
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed, otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke(
    {"input": "Where is its headquartered?", "chat_history": chat_history}
)

'Where is GreenGrow Innovations headquartered?'

In [42]:
# create and invoke a history-aware retriever
from langchain.chains import create_history_aware_retriever

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt,
)
history_aware_retriever.invoke(
    {"input": "Where is its headquartered?", "chat_history": chat_history}
)

[Document(id='501eaaeb-ea42-4d73-aa60-147a8ab65f68', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in vertical farming, drought-resistant crop development, and AI-powered farm management systems.\n\n\n\nDespite its growth, GreenGrow remains committed to its original mission of promoting sustainable farming practices. The company regularly partners with universities and research institutions

In [43]:
# Invoke the history-UNAWARE retriever with a follow-up question
retriever.invoke("Where is it headquartered?")

[Document(id='d59d8161-fcdf-4f15-9029-be1683654740', metadata={'source': 'docs\\Company_ QuantumNext Systems.docx'}, page_content='Company: QuantumNext Systems\n\nHeadquarters: QuantumNext Systems is headquartered in Bangalore, Karnataka, India. The company, specializing in quantum computing and advanced data processing, is situated in the bustling tech metropolis of Bangalore, often referred to as the "Silicon Valley of India." From this technology capital, QuantumNext Systems is well-positioned to tap into India\'s rich pool of engineering talent and growing tech ecosystem, enabling it to push the boundaries of computational innovation.'),
 Document(id='59c99541-4bbf-4626-9d7b-d9d879a23456', metadata={'source': 'docs\\Company_ TechWave Innovations.docx'}, page_content='Company: TechWave Innovations\n\nHeadquarters: TechWave Innovations is headquartered in San Francisco, California, USA. As a leader in cutting-edge AI and machine learning solutions, the company thrives in the heart of

In [44]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant. Use the following context to answer the user's question.",
        ),
        ("system", "Context: {context}"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [45]:
rag_chain.invoke({"input": "Where it is headquartered?", "chat_history": chat_history})

{'input': 'Where it is headquartered?',
 'chat_history': [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={})],
 'context': [Document(id='501eaaeb-ea42-4d73-aa60-147a8ab65f68', metadata={'source': 'docs\\GreenGrow Innovations_ Company History.docx'}, page_content="The company's breakthrough came in 2018 with the introduction of the EcoHarvest System, an integrated solution that combined smart irrigation, soil monitoring, and automated harvesting techniques. This system caught the attention of large-scale farmers across the United States, propelling GreenGrow to national prominence.\n\n\n\nToday, GreenGrow Innovations employs over 200 people and has expanded its operations to include offices in California and Iowa. The company continues to focus on developing sustainable agricultural technologies, with ongoing projects in 

### Building Multi User Chatbot

In [46]:
import sqlite3

DB_NAME = "rag_app.db"


def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn


def create_application_logs():
    conn = get_db_connection()
    conn.execute("""CREATE TABLE IF NOT EXISTS application_logs
                    (id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT,
                    user_query TEXT,
                    gpt_response TEXT,
                    model TEXT,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)
                    """)
    conn.close()


# Function to insert application logs into the database
def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute(
        "INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?, ?, ?, ?)",
        (session_id, user_query, gpt_response, model),
    )
    conn.commit()
    conn.close()


# Function to retrieve chat history for a given session ID
# This function fetches user queries and GPT responses from the database
def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT user_query, gpt_response FROM application_logs WHERE session_id = ? ORDER BY created_at",
        (session_id,),
    )
    messages = []
    for row in cursor.fetchall():
        messages.extend(
            [
                {"role": "human", "content": row["user_query"]},
                {"role": "ai", "content": row["gpt_response"]},
            ]
        )
    conn.close()
    return messages


# Initialize the database
create_application_logs()

In [47]:
import uuid

session_id = str(uuid.uuid4())
chat_history = get_chat_history(session_id)
print(chat_history)

question1 = "When was GreenGrow Innovations founded?"
answer1 = rag_chain.invoke({"input": question1, "chat_history": chat_history})["answer"]
insert_application_logs(session_id, question1, answer1, "gpt-4-o-mini")

print(f"Human: {question1}")
print(f"AI: {answer1}\n")

[]
Human: When was GreenGrow Innovations founded?
AI: GreenGrow Innovations was founded in 2010.



In [48]:
question2 = "Where it is headquartered?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer2 = rag_chain.invoke({"input": question2, "chat_history": chat_history})["answer"]
insert_application_logs(session_id, question2, answer2, "gpt-3.5-turbo")
print(f"Human: {question2}")
print(f"AI: {answer2}\n")

[{'role': 'human', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'ai', 'content': 'GreenGrow Innovations was founded in 2010.'}]
Human: Where it is headquartered?
AI: GreenGrow Innovations is headquartered in Portland, Oregon.



In [49]:
question3 = "On what continent is it located?"
chat_history = get_chat_history(session_id)
print(chat_history)
answer3 = rag_chain.invoke({"input": question3, "chat_history": chat_history})["answer"]
insert_application_logs(session_id, question3, answer3, "gpt-3.5-turbo")
print(f"Human: {question3}")
print(f"AI: {answer3}\n")

[{'role': 'human', 'content': 'When was GreenGrow Innovations founded?'}, {'role': 'ai', 'content': 'GreenGrow Innovations was founded in 2010.'}, {'role': 'human', 'content': 'Where it is headquartered?'}, {'role': 'ai', 'content': 'GreenGrow Innovations is headquartered in Portland, Oregon.'}]
Human: On what continent is it located?
AI: GreenGrow Innovations is located on the continent of North America.



New User

In [50]:
session_id = str(uuid.uuid4())
question = "What is GreenGrow"
chat_history = get_chat_history(session_id)
print(chat_history)
answer = rag_chain.invoke({"input": question, "chat_history": chat_history})["answer"]
insert_application_logs(session_id, question, answer, "gpt-3.5-turbo")
print(f"Human: {question}")
print(f"AI: {answer}\n")

[]
Human: What is GreenGrow
AI: GreenGrow Innovations is a company founded in 2010 by agricultural engineers Sarah Chen and Michael Rodriguez. It focuses on developing sustainable farming technologies to make agriculture more environmentally friendly and efficient. The company initially started in a small garage in Portland, Oregon, with its first product, the WaterWise Sensor, launched in 2012 to help reduce water usage in farming.

Over the years, GreenGrow has expanded its product offerings and operations, with notable products including the SoilHealth Monitor and the EcoHarvest System, which integrates smart irrigation, soil monitoring, and automated harvesting. By 2023, the company employs over 200 people and has expanded to offices in California and Iowa, while continuing to engage in research on vertical farming, drought-resistant crops, and AI-powered farm management systems. GreenGrow remains committed to its mission of advancing sustainable agricultural practices through part